# LIMINA -- 03. Pelatihan Model

Melatih dari `data/panel.csv`, salah satu dari tiga jalur tergantung data
lengkap (data_complete=1) yang tersedia:

1. **Kandidat 1+2** (regresi logistik + gradient boosting) -- kalau >=20
   baris lengkap dan >=2 positif.
2. **Kandidat 5** (anomali tanpa label, Isolation Forest) -- kalau positif
   kurang tapi >=5 baris lengkap. Tidak butuh peristiwa suspensi sama
   sekali, hanya menandai emiten yang beda dari mayoritas peer-nya.
3. **rule_based_penuh** -- kalau baris lengkap < 5. Notebook 05 otomatis
   memakai Kandidat 4 (baselines.py).

Tidak ada jalur yang mem-fabrikasi data atau melempar error tak tertangani;
keputusan dicatat ke `artifacts/keputusan.json`.

In [1]:
import sys
from pathlib import Path


def _cari_root(mulai: Path) -> Path:
    for kandidat in [mulai, *mulai.parents]:
        if (kandidat / "limina" / "__init__.py").exists():
            return kandidat
    raise RuntimeError("Folder 'limina/' tidak ditemukan -- jalankan dari dalam proyek LIMINA.")


ROOT = _cari_root(Path.cwd())
sys.path.insert(0, str(ROOT))

import json

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from limina import config, contracts, leakage, model_registry, models, snapshot, splits

panel = pd.read_csv(config.PATH_PANEL, parse_dates=["as_of_date", "event_date", "feature_max_source_date"])
with open(config.PATH_JENDELA) as f:
    jendela = json.load(f)
cutoff_latih = pd.Timestamp(jendela["cutoff_latih"])

print(f"Panel: {len(panel)} baris, cutoff latih: {cutoff_latih.date()}")

Panel: 260 baris, cutoff latih: 2026-03-28


## 1. Pisahkan data latih, tentukan jalur

In [2]:
train_data = splits.pisahkan_temporal(panel, cutoff_latih)
df_lengkap = train_data[train_data["data_complete"] == 1].reset_index(drop=True)
positif = int(df_lengkap["is_event_90d"].sum()) if len(df_lengkap) else 0

JALUR = "supervised" if len(df_lengkap) >= 20 and positif >= 2 else \
        "anomali" if len(df_lengkap) >= 5 else \
        "rule_based"

print(f"Data lengkap: {len(df_lengkap)} baris, {positif} positif -> jalur: {JALUR}")

if JALUR == "supervised":
    X_train, y_train = df_lengkap[contracts.KOLOM_FITUR], df_lengkap["is_event_90d"]
    median_latih = X_train.median()
elif JALUR == "anomali":
    X_train = df_lengkap[contracts.KOLOM_FITUR]
    median_latih = X_train.median()

Data lengkap: 11 baris, 0 positif -> jalur: anomali


## 2. Latih model sesuai jalur

In [3]:
if JALUR == "supervised":
    model_lr, scaler = models.latih_kandidat_1(X_train, y_train)
    model_gb = models.latih_kandidat_2(X_train, y_train)
    print("Kandidat 1+2 dilatih.")
elif JALUR == "anomali":
    model_iso = models.latih_kandidat_5(X_train)
    print("Kandidat 5 (anomali) dilatih -- tidak pakai label suspensi.")
else:
    print("Baris lengkap < 5, tidak ada model yang dilatih.")

Kandidat 5 (anomali) dilatih -- tidak pakai label suspensi.


## 3. Pemeriksaan kebocoran (hanya jalur supervised)

Kalau gagal, notebook berhenti di sini -- hasil terlalu bagus adalah
gejala, bukan kabar baik (`AMBANG-peran-model-dan-evaluasi.md` bagian 6).

In [4]:
if JALUR == "supervised":
    X_train_filled = X_train.fillna(median_latih)
    X_train_scaled = scaler.transform(X_train_filled)
    hasil_kebocoran = leakage.jalankan_semua_pemeriksaan(
        df_panel=train_data, model_class=LogisticRegression, scaler_class=StandardScaler,
        X_train=X_train_filled, y_train=y_train, X_train_scaled=X_train_scaled, lempar_error=True,
    )
    print(f"Lolos. AUC pengacakan label: {hasil_kebocoran['auc_pengacakan_label']:.3f} (target ~0.50)")
    with open(config.ARTIFACTS_DIR / "hasil_kebocoran.json", "w", encoding="utf-8") as f:
        json.dump(hasil_kebocoran, f, indent=2)
else:
    print("Dilewati -- hanya berlaku untuk jalur supervised.")

Dilewati -- hanya berlaku untuk jalur supervised.


## 4. Simpan model / catat keputusan

In [5]:
config.ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

if JALUR == "supervised":
    stempel = model_registry.simpan_model_terlatih(model_lr, scaler, median_latih, model_gb=model_gb)
    print(f"Model disimpan, versi: {stempel}")
elif JALUR == "anomali":
    stempel = model_registry.simpan_model_kandidat_5(model_iso, median_latih)
    keputusan = snapshot.keputusan_anomali_saja(f"{positif} positif dari {len(df_lengkap)} baris lengkap")
    with open(config.PATH_KEPUTUSAN, "w", encoding="utf-8") as f:
        json.dump(keputusan, f, indent=2)
    print(f"Kandidat 5 disimpan, versi: {stempel}. Keputusan: {keputusan['keputusan']}")
else:
    keputusan = snapshot.keputusan_data_tidak_cukup(f"{len(df_lengkap)} baris lengkap, {positif} positif")
    with open(config.PATH_KEPUTUSAN, "w", encoding="utf-8") as f:
        json.dump(keputusan, f, indent=2)
    print(f"Keputusan: {keputusan['keputusan']} -- {keputusan['alasan']}")

print("Lanjut ke notebook 05 (penilaian_dan_artefak).")

Kandidat 5 disimpan, versi: 20260924_001142Z. Keputusan: anomali_tanpa_label
Lanjut ke notebook 05 (penilaian_dan_artefak).
